# LeetCode 715 — Range Module bằng Python

- Link: https://leetcode.com/problems/range-module/
- Độ khó: **Hard**
- Chủ đề: Thiết kế cấu trúc dữ liệu, Khoảng nửa mở, Gộp khoảng (merge intervals), Tìm kiếm nhị phân (`bisect`), Segment Tree động

Skeleton trên LeetCode (Python3):

```python
class RangeModule:

    def __init__(self):

    def addRange(self, left: int, right: int) -> None:

    def queryRange(self, left: int, right: int) -> bool:

    def removeRange(self, left: int, right: int) -> None:


# Your RangeModule object will be instantiated and called as such:
# obj = RangeModule()
# obj.addRange(left,right)
# param_2 = obj.queryRange(left,right)
# obj.removeRange(left,right)
```

## 1. Giới thiệu & mục tiêu học

Ta cần một "bộ quản lý khoảng": thêm khoảng, xoá khoảng, và hỏi "khoảng này có được phủ **kín** không?". Toạ độ tới `10^9` nên không thể dùng mảng đánh dấu từng điểm.

Đây là bài thiết kế kinh điển cho kỹ thuật **duy trì tập các khoảng rời nhau đã sắp xếp**. Ta sẽ thấy một lời giải chỉ ~15 dòng dùng `bisect` trên một **danh sách mốc phẳng** với mẹo **chẵn/lẻ** — rất đẹp nhưng cần hiểu kỹ mới viết đúng.

Sau notebook này bạn sẽ:

1. Làm chủ khoảng **nửa mở** `[left, right)`: khi nào hai khoảng chạm nhau thì gộp, khi nào thì không.
2. Viết mô phỏng "từng ô đơn vị" làm trọng tài.
3. Duy trì danh sách khoảng rời nhau: gộp khi thêm, cắt khi xoá, tra cứu khi hỏi.
4. Hiểu mẹo "danh sách mốc phẳng + chẵn/lẻ" với `bisect_left`/`bisect_right`.
5. Cài Segment Tree **động** (tạo nút khi cần) cho toạ độ lớn, và kiểm chứng mọi cách bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Một Range Module theo dõi các khoảng số thực. Khoảng nửa mở `[left, right)` gồm mọi số thực `x` với `left <= x < right`. Cài đặt lớp `RangeModule`:

- `RangeModule()`: khởi tạo, chưa theo dõi gì.
- `addRange(left, right)`: bắt đầu theo dõi **mọi** số thực trong `[left, right)`. Phần nào đã được theo dõi thì vẫn giữ.
- `queryRange(left, right)`: trả về `True` nếu **mọi** số thực trong `[left, right)` đang được theo dõi, ngược lại `False`.
- `removeRange(left, right)`: ngừng theo dõi mọi số thực trong `[left, right)` đang được theo dõi.

**Ví dụ**

```text
Input:
["RangeModule", "addRange", "removeRange", "queryRange", "queryRange", "queryRange"]
[[], [10, 20], [14, 16], [10, 14], [13, 15], [16, 17]]
Output:
[null, null, null, true, false, true]
Giải thích:
  addRange(10, 20)    -> theo dõi [10, 20)
  removeRange(14, 16) -> còn [10, 14) và [16, 20)
  queryRange(10, 14)  -> true  (phủ kín bởi [10, 14))
  queryRange(13, 15)  -> false (14 .. 15 không được theo dõi)
  queryRange(16, 17)  -> true  (vẫn trong [16, 20))
```

**Ràng buộc**

- `1 <= left < right <= 10^9`
- Tổng số lần gọi `addRange`, `queryRange`, `removeRange` tối đa `10^4`.

In [1]:
def run(cls, ops, args):
    """Chạy chuỗi lệnh theo định dạng LeetCode, trả về list kết quả."""
    out, obj = [], None
    for op, a in zip(ops, args):
        if op == "RangeModule":
            obj = cls()
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

EX_OPS = ["RangeModule", "addRange", "removeRange", "queryRange", "queryRange", "queryRange"]
EX_ARGS = [[], [10, 20], [14, 16], [10, 14], [13, 15], [16, 17]]
EX_OUT = [None, None, None, True, False, True]
print(list(zip(EX_OPS, EX_ARGS, EX_OUT)))

[('RangeModule', [], None), ('addRange', [10, 20], None), ('removeRange', [14, 16], None), ('queryRange', [10, 14], True), ('queryRange', [13, 15], False), ('queryRange', [16, 17], True)]


## 3. Kiến thức nền

### 3.1. Khoảng nửa mở: chạm nhau và chồng nhau

Với `[a, b)` và `[c, d)` (giả sử `a <= c`):

- **Chồng nhau** (chung độ dài dương) khi `c < b`.
- **Chạm nhau** khi `c == b` — không chung điểm nào, nhưng **hợp** của chúng là một khoảng liền `[a, d)`. Vì vậy khi **thêm**, ta nên **gộp** cả hai trường hợp (`c <= b`) để danh sách gọn và truy vấn đúng.

In [2]:
def union_if_touch(a, b, c, d):
    """Trả về hợp nếu hai khoảng [a,b), [c,d) (a <= c) liền nhau hoặc chồng nhau, ngược lại None."""
    return (a, max(b, d)) if c <= b else None

print(union_if_touch(1, 3, 3, 5))   # chạm nhau -> (1, 5)
print(union_if_touch(1, 3, 2, 8))   # chồng nhau -> (1, 8)
print(union_if_touch(1, 3, 4, 5))   # rời nhau -> None

(1, 5)
(1, 8)
None


### 3.2. Danh sách khoảng rời nhau đã sắp xếp

Ta lưu `[[a0, b0], [a1, b1], ...]` với `a0 < b0 < a1 < b1 < ...` (**nghiêm ngặt**: hai khoảng liền nhau đã được gộp). Khi đó:

> `[left, right)` được phủ kín ⇔ có **một** khoảng `[a, b)` với `a <= left` và `right <= b`.

(Vì giữa hai khoảng liên tiếp luôn có khe hở dương.)

### 3.3. `bisect_left` và `bisect_right` khi có phần tử bằng

Với mảng tăng `xs`:

- `bisect_left(xs, x)`: số phần tử **< x**.
- `bisect_right(xs, x)`: số phần tử **≤ x**.

Khác nhau chỉ khi `x` có mặt trong `xs` — và đó chính là chỗ quyết định "chạm nhau" trong mẹo mốc phẳng.

In [3]:
from bisect import bisect_left, bisect_right
xs = [10, 14, 16, 20]
for x in [9, 10, 12, 14, 20, 25]:
    print(f"x={x:>2}: bisect_left={bisect_left(xs, x)}, bisect_right={bisect_right(xs, x)}")

x= 9: bisect_left=0, bisect_right=0
x=10: bisect_left=0, bisect_right=1
x=12: bisect_left=1, bisect_right=1
x=14: bisect_left=1, bisect_right=2
x=20: bisect_left=3, bisect_right=4
x=25: bisect_left=4, bisect_right=4


### 3.4. Gán lát cắt `xs[i:j] = new`

Thay đoạn `xs[i:j]` bằng list `new` (độ dài có thể khác). Nếu `i == j` thì đó là **chèn** vào vị trí `i`.

In [4]:
xs = [10, 20]
xs[1:1] = [14, 16]          # chèn
print(xs)
xs[1:3] = []                # xoá
print(xs)

[10, 14, 16, 20]
[10, 20]


### 3.5. `__slots__` cho lớp nút nhỏ

Khi tạo rất nhiều đối tượng nhỏ (nút cây), khai báo `__slots__` giúp tiết kiệm bộ nhớ và truy cập thuộc tính nhanh hơn.

In [5]:
class Node:
    __slots__ = ("left", "right", "cov")
    def __init__(self, cov=False):
        self.left = self.right = None
        self.cov = cov

nd = Node(True)
print(nd.cov, nd.left)

True None


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Trạng thái = hợp của các khoảng rời nhau

Dù thêm/xoá theo thứ tự nào, tập số đang được theo dõi luôn là **hợp hữu hạn các khoảng nửa mở rời nhau**. Ta chỉ cần lưu các khoảng đó ở dạng chuẩn (sắp xếp, gộp các khoảng chạm nhau).

### 4.2. Tính tay ví dụ

| Lệnh | Danh sách khoảng sau lệnh | Kết quả |
|---|---|---|
| `addRange(10, 20)` | `[10,20)` | — |
| `removeRange(14, 16)` | `[10,14)`, `[16,20)` | — |
| `queryRange(10, 14)` | (không đổi) | `[10,14) ⊆ [10,14)` → **true** |
| `queryRange(13, 15)` | (không đổi) | không khoảng nào chứa cả `[13,15)` → **false** |
| `queryRange(16, 17)` | (không đổi) | `[16,17) ⊆ [16,20)` → **true** |

### 4.3. Thêm một ví dụ có gộp

`add(1,3)`, `add(5,7)`, `add(3,5)` → các khoảng `[1,3)`, `[5,7)` chạm `[3,5)` ở cả hai đầu → gộp thành `[1,7)`. Sau đó `query(2, 6)` → **true**. Nếu ta **không** gộp các khoảng chạm nhau thì truy vấn "một khoảng chứa trọn" sẽ trả sai.

In [6]:
intervals = [[1, 3], [5, 7]]
new = [3, 5]
merged = sorted(intervals + [new])
res = [merged[0]]
for a, b in merged[1:]:
    if a <= res[-1][1]:
        res[-1][1] = max(res[-1][1], b)
    else:
        res.append([a, b])
print("sau khi gộp:", res)

sau khi gộp: [[1, 7]]


## 5. Cách 1 — Brute Force: tập các ô đơn vị

**Ý tưởng.** Với toạ độ nguyên nhỏ, khoảng `[l, r)` = hợp các ô `[x, x+1)` với `x = l .. r-1`. Lưu tập các `x` đang được theo dõi.

(Vì mọi `left`, `right` đều nguyên, "mọi số thực trong `[l, r)`" được theo dõi ⇔ mọi ô `[x, x+1)` trong đó được theo dõi.)

In [7]:
class RangeModuleBrute:
    def __init__(self):
        self.cells = set()

    def addRange(self, left, right):
        self.cells.update(range(left, right))

    def queryRange(self, left, right):
        return all(x in self.cells for x in range(left, right))

    def removeRange(self, left, right):
        self.cells.difference_update(range(left, right))

print(run(RangeModuleBrute, EX_OPS, EX_ARGS))

[None, None, None, True, False, True]


**Độ phức tạp.** Mỗi lệnh `O(right - left)`, bộ nhớ `O(tổng độ dài được theo dõi)`. Với toạ độ `10^9`, `addRange(1, 10^9)` cần một tỷ phần tử → không thể.

**Nút thắt.** Ta lưu từng **điểm** trong khi chỉ cần lưu các **mút** của khoảng. Số mút ≤ `2 ×` số lệnh.

## 6. Bước đột phá

**Insight 1 — chỉ lưu mút.** Lưu danh sách khoảng rời nhau, đã sắp xếp, các khoảng chạm nhau đã gộp. Mỗi lệnh chỉ ảnh hưởng tới các khoảng giao/chạm với `[left, right)`.

**Insight 2 — danh sách mốc phẳng + chẵn/lẻ.** Viết các khoảng thành một list phẳng `track = [a0, b0, a1, b1, ...]` (tăng nghiêm ngặt). Một điểm `x` được theo dõi ⇔ **số mốc ≤ x là số lẻ** (đã đi qua một "mở" mà chưa gặp "đóng").

- `i = bisect_left(track, left)` = số mốc `< left`. **Nếu `i` chẵn** thì `left` nằm **ngoài** mọi khoảng (hoặc đúng ở một mút trái); **nếu lẻ** thì `left` nằm trong một khoảng (hoặc đúng ở mút phải — tức chạm).
- `j = bisect_right(track, right)` = số mốc `≤ right`. **Nếu `j` chẵn** thì `right` nằm ngoài (hoặc đúng mút phải); **nếu lẻ** thì nằm trong (hoặc đúng mút trái của khoảng sau — chạm).

Quy tắc "nếu… thì…" cho `addRange`: thay `track[i:j]` bằng:

- `[left]` **nếu** `i` chẵn (left là mút trái mới), không thêm gì nếu lẻ (left bị "nuốt" vào khoảng đang mở);
- cộng `[right]` **nếu** `j` chẵn (right là mút phải mới), không thêm gì nếu lẻ.

`removeRange` đối xứng: thêm `left` **nếu `i` lẻ** (cắt khoảng đang mở tại left), thêm `right` **nếu `j` lẻ**.

`queryRange`: `i = bisect_right(track, left)`, `j = bisect_left(track, right)`; phủ kín ⇔ `i == j` (không có mốc nào nằm trong `(left, right)`... chính xác hơn, không có mốc trong `[left, right)` làm gián đoạn) **và** `i` lẻ (đang ở trong một khoảng).

**Insight 3 — Segment Tree động.** Coi trục `[1, 10^9)` là `10^9` ô; nút cây lưu "đoạn của tôi có được phủ kín không". Gán cả đoạn = phủ/không phủ; truy vấn = AND trên đoạn. Chỉ tạo nút khi cần → `O(log C)` nút mỗi lệnh.

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

| Cách | Cấu trúc | add/remove | query |
|---|---|---|---|
| 1 | Tập ô đơn vị | `O(r - l)` | `O(r - l)` |
| 2 | List khoảng rời nhau, duyệt tuyến tính | `O(k)` | `O(log k)` |
| 3 | List mốc phẳng + `bisect` + chẵn/lẻ | `O(k)` (gán lát cắt), tìm `O(log k)` | `O(log k)` |
| 4 | Segment Tree động | `O(log C)` | `O(log C)` |

(`k` = số khoảng hiện có, `C = 10^9`.)

### 7.1. Cách 2 — Danh sách khoảng, duyệt tuyến tính

In [8]:
class RangeModuleList:
    def __init__(self):
        self.iv = []                                   # [[a, b], ...] rời nhau, tăng dần, đã gộp chạm

    def addRange(self, left, right):
        res, placed = [], False
        for a, b in self.iv:
            if b < left:                               # nằm hẳn bên trái (không chạm)
                res.append([a, b])
            elif right < a:                            # nằm hẳn bên phải (không chạm)
                if not placed:
                    res.append([left, right]); placed = True
                res.append([a, b])
            else:                                      # chồng hoặc chạm -> gộp
                left, right = min(left, a), max(right, b)
        if not placed:
            res.append([left, right])
        self.iv = res

    def queryRange(self, left, right):
        k = bisect_right(self.iv, [left, float('inf')]) - 1   # khoảng có a <= left lớn nhất
        return k >= 0 and self.iv[k][0] <= left and right <= self.iv[k][1]

    def removeRange(self, left, right):
        res = []
        for a, b in self.iv:
            if b <= left or right <= a:                # không chồng (chạm cũng không ảnh hưởng)
                res.append([a, b])
            else:
                if a < left:
                    res.append([a, left])              # phần còn lại bên trái
                if right < b:
                    res.append([right, b])             # phần còn lại bên phải
        self.iv = res

print(run(RangeModuleList, EX_OPS, EX_ARGS))
rm = RangeModuleList()
for l, r in [(1, 3), (5, 7), (3, 5)]:
    rm.addRange(l, r)
print("sau add (1,3),(5,7),(3,5):", rm.iv, "| query(2,6):", rm.queryRange(2, 6))

[None, None, None, True, False, True]
sau add (1,3),(5,7),(3,5): [[1, 7]] | query(2,6): True


Giải thích các điều kiện:

- **add**: khoảng `[a, b)` được giữ nguyên khi `b < left` (kết thúc **trước** left, không chạm) hoặc `right < a`. Trường hợp `b == left` hay `a == right` (chạm) rơi vào nhánh gộp.
- **remove**: dùng `b <= left or right <= a` — chạm thì **không** bị ảnh hưởng (không có điểm chung).
- **query**: `bisect_right(self.iv, [left, inf]) - 1` tìm khoảng có mút trái `≤ left` lớn nhất (so sánh list theo từ điển); chỉ khoảng đó mới có thể chứa `[left, right)`.

**Code nộp LeetCode (Cách 2):**

```python
from bisect import bisect_right

class RangeModule:
    def __init__(self):
        self.iv = []

    def addRange(self, left: int, right: int) -> None:
        res, placed = [], False
        for a, b in self.iv:
            if b < left:
                res.append([a, b])
            elif right < a:
                if not placed:
                    res.append([left, right]); placed = True
                res.append([a, b])
            else:
                left, right = min(left, a), max(right, b)
        if not placed:
            res.append([left, right])
        self.iv = res

    def queryRange(self, left: int, right: int) -> bool:
        k = bisect_right(self.iv, [left, float('inf')]) - 1
        return k >= 0 and self.iv[k][0] <= left and right <= self.iv[k][1]

    def removeRange(self, left: int, right: int) -> None:
        res = []
        for a, b in self.iv:
            if b <= left or right <= a:
                res.append([a, b])
            else:
                if a < left: res.append([a, left])
                if right < b: res.append([right, b])
        self.iv = res
```

**Độ phức tạp.** add/remove `O(k)` (dựng lại list), query `O(log k)`; bộ nhớ `O(k)`. Với `10^4` lệnh, `k ≤ 10^4` → tệ nhất `10^8` — hơi nặng, nhưng thực tế số khoảng thường nhỏ hơn nhiều.

### 7.2. Cách 3 — Danh sách mốc phẳng + `bisect` + chẵn/lẻ

In [9]:
class RangeModuleBisect:
    def __init__(self):
        self.track = []                      # [a0, b0, a1, b1, ...] tăng nghiêm ngặt

    def addRange(self, left, right):
        i = bisect_left(self.track, left)
        j = bisect_right(self.track, right)
        sub = []
        if i % 2 == 0:
            sub.append(left)
        if j % 2 == 0:
            sub.append(right)
        self.track[i:j] = sub

    def queryRange(self, left, right):
        i = bisect_right(self.track, left)
        j = bisect_left(self.track, right)
        return i == j and i % 2 == 1

    def removeRange(self, left, right):
        i = bisect_left(self.track, left)
        j = bisect_right(self.track, right)
        sub = []
        if i % 2 == 1:
            sub.append(left)
        if j % 2 == 1:
            sub.append(right)
        self.track[i:j] = sub

print(run(RangeModuleBisect, EX_OPS, EX_ARGS))
rm = RangeModuleBisect()
for l, r in [(1, 3), (5, 7), (3, 5)]:
    rm.addRange(l, r)
print("track sau add (1,3),(5,7),(3,5):", rm.track, "| query(2,6):", rm.queryRange(2, 6))

[None, None, None, True, False, True]
track sau add (1,3),(5,7),(3,5): [1, 7] | query(2,6): True


Giải thích từng lệnh:

- **add**: các mốc trong `track[i:j]` là mọi mốc nằm trong `[left, right]` (kể cả mốc **bằng** `left` nhờ `bisect_left`, và mốc **bằng** `right` nhờ `bisect_right`) — chúng đều bị khoảng mới "nuốt". Thay bằng `left` nếu `left` đang ở ngoài (`i` chẵn) và `right` nếu `right` đang ở ngoài (`j` chẵn). Mốc bằng `left` là mút **phải** của khoảng trước (khi `i` lẻ) → bị xoá → **gộp chạm** tự động.
- **remove**: ngược lại, chèn `left` nếu nó đang ở **trong** một khoảng (`i` lẻ → cắt khoảng đó tại `left`), chèn `right` nếu `j` lẻ.
- **query**: `bisect_right(left)` = số mốc `≤ left`; `bisect_left(right)` = số mốc `< right`. Bằng nhau ⇔ không có mốc nào trong `(left, right)`; nghĩa là cả `[left, right)` nằm trong **cùng một** đoạn giữa hai mốc liên tiếp. Đoạn đó được theo dõi ⇔ số mốc trước nó lẻ.

**Code nộp LeetCode (Cách 3):**

```python
from bisect import bisect_left, bisect_right

class RangeModule:
    def __init__(self):
        self.track = []

    def addRange(self, left: int, right: int) -> None:
        i, j = bisect_left(self.track, left), bisect_right(self.track, right)
        self.track[i:j] = [left] * (i % 2 == 0) + [right] * (j % 2 == 0)

    def queryRange(self, left: int, right: int) -> bool:
        i, j = bisect_right(self.track, left), bisect_left(self.track, right)
        return i == j and i % 2 == 1

    def removeRange(self, left: int, right: int) -> None:
        i, j = bisect_left(self.track, left), bisect_right(self.track, right)
        self.track[i:j] = [left] * (i % 2 == 1) + [right] * (j % 2 == 1)
```

**Độ phức tạp.** Tìm vị trí `O(log k)`; gán lát cắt `O(k)` tệ nhất (dời phần đuôi list, chạy bằng C nên rất nhanh); query `O(log k)`. Bộ nhớ `O(k)`.

### 7.3. Cách 4 — Segment Tree động

**Giải thích.** Cây quản lý các ô `[1, C]` với `C = 10^9` (ô `x` = `[x, x+1)`). Mỗi nút lưu `cov` (đoạn của nút có được phủ **kín** không). Nút **lá logic** (không có con) nghĩa là cả đoạn **đồng nhất** với giá trị `cov`.

- **update(l, r, val)** trên ô `l .. r-1`: nếu đoạn của nút nằm trọn trong vùng → `cov = val`, **bỏ con** (đoạn đồng nhất). Nếu giao một phần → nếu đang là lá thì tạo hai con mang `cov` hiện tại ("đẩy xuống"), đệ quy, rồi `cov = left.cov and right.cov`.
- **query(l, r)**: nằm ngoài → `True` (không ảnh hưởng AND); nằm trọn hoặc nút là lá → `cov`; còn lại AND hai con.

In [10]:
class RangeModuleSegTree:
    LO, HI = 1, 10**9                      # các ô 1 .. 10^9 - 1 (dùng HI làm cận trên an toàn)

    class _Node:
        __slots__ = ("l", "r", "cov")
        def __init__(self, cov=False):
            self.l = self.r = None
            self.cov = cov

    def __init__(self):
        self.root = self._Node(False)

    def _update(self, node, lo, hi, ql, qr, val):
        if qr < lo or hi < ql:
            return
        if ql <= lo and hi <= qr:
            node.cov = val
            node.l = node.r = None         # đoạn đồng nhất -> bỏ con
            return
        if node.l is None:                 # đang đồng nhất -> tách ra hai con cùng giá trị
            node.l, node.r = self._Node(node.cov), self._Node(node.cov)
        mid = (lo + hi) // 2
        self._update(node.l, lo, mid, ql, qr, val)
        self._update(node.r, mid + 1, hi, ql, qr, val)
        node.cov = node.l.cov and node.r.cov

    def _query(self, node, lo, hi, ql, qr):
        if qr < lo or hi < ql:
            return True
        if (ql <= lo and hi <= qr) or node.l is None:
            return node.cov
        mid = (lo + hi) // 2
        return self._query(node.l, lo, mid, ql, qr) and self._query(node.r, mid + 1, hi, ql, qr)

    def addRange(self, left, right):
        self._update(self.root, self.LO, self.HI, left, right - 1, True)

    def queryRange(self, left, right):
        return self._query(self.root, self.LO, self.HI, left, right - 1)

    def removeRange(self, left, right):
        self._update(self.root, self.LO, self.HI, left, right - 1, False)

print(run(RangeModuleSegTree, EX_OPS, EX_ARGS))

[None, None, None, True, False, True]


**Vì sao "nút lá = đoạn đồng nhất" đúng khi truy vấn?** Nếu nút không có con, mọi ô trong đoạn của nó có cùng trạng thái `cov`, nên phần giao với truy vấn cũng vậy → trả `cov`.

**Code nộp LeetCode (Cách 4):**

```python
class RangeModule:
    class Node:
        __slots__ = ("l", "r", "cov")
        def __init__(self, cov=False):
            self.l = self.r = None; self.cov = cov

    def __init__(self):
        self.root = self.Node(False)

    def _upd(self, nd, lo, hi, ql, qr, val):
        if qr < lo or hi < ql: return
        if ql <= lo and hi <= qr:
            nd.cov = val; nd.l = nd.r = None; return
        if nd.l is None:
            nd.l, nd.r = self.Node(nd.cov), self.Node(nd.cov)
        mid = (lo + hi) // 2
        self._upd(nd.l, lo, mid, ql, qr, val)
        self._upd(nd.r, mid + 1, hi, ql, qr, val)
        nd.cov = nd.l.cov and nd.r.cov

    def _qry(self, nd, lo, hi, ql, qr):
        if qr < lo or hi < ql: return True
        if (ql <= lo and hi <= qr) or nd.l is None: return nd.cov
        mid = (lo + hi) // 2
        return self._qry(nd.l, lo, mid, ql, qr) and self._qry(nd.r, mid + 1, hi, ql, qr)

    def addRange(self, left: int, right: int) -> None:
        self._upd(self.root, 1, 10**9, left, right - 1, True)

    def queryRange(self, left: int, right: int) -> bool:
        return self._qry(self.root, 1, 10**9, left, right - 1)

    def removeRange(self, left: int, right: int) -> None:
        self._upd(self.root, 1, 10**9, left, right - 1, False)
```

**Độ phức tạp.** Mỗi lệnh đi qua `O(log C)` ≈ 30 tầng, tạo tối đa `O(log C)` nút mới → `O(log C)` thời gian; bộ nhớ `O(q log C)` tệ nhất (`q` = số lệnh), giảm nhờ bỏ con khi đoạn đồng nhất. Ưu điểm lớn: mở rộng dễ cho truy vấn khác (ví dụ "tổng độ dài được phủ").

## 8. TRACE từng bước

### 8.1. Bảng trạng thái Cách 3 (mốc phẳng) cho ví dụ

| Lệnh | `track` trước | `i` | `j` | chẵn/lẻ | thay `track[i:j]` bằng | `track` sau | kết quả |
|---|---|---|---|---|---|---|---|
| add(10,20) | `[]` | 0 | 0 | i chẵn, j chẵn | `[10, 20]` | `[10, 20]` | |
| remove(14,16) | `[10, 20]` | 1 | 1 | i lẻ, j lẻ | `[14, 16]` | `[10, 14, 16, 20]` | |
| query(10,14) | | `bisect_right(10)=1` | `bisect_left(14)=1` | bằng nhau, lẻ | | | **true** |
| query(13,15) | | 1 | 2 | khác nhau | | | **false** |
| query(16,17) | | 3 | 3 | bằng nhau, lẻ | | | **true** |

### 8.2. `trace_bisect`

In [11]:
def trace_bisect(ops):
    track = []
    for op, l, r in ops:
        if op == "add":
            i, j = bisect_left(track, l), bisect_right(track, r)
            sub = [l] * (i % 2 == 0) + [r] * (j % 2 == 0)
            before = track[:]
            track[i:j] = sub
            print(f"add({l},{r}):    i={i} ({'chẵn' if i % 2 == 0 else 'lẻ'}), j={j} ({'chẵn' if j % 2 == 0 else 'lẻ'}), "
                  f"thay {before[i:j]} bằng {sub} -> {track}")
        elif op == "remove":
            i, j = bisect_left(track, l), bisect_right(track, r)
            sub = [l] * (i % 2 == 1) + [r] * (j % 2 == 1)
            before = track[:]
            track[i:j] = sub
            print(f"remove({l},{r}): i={i} ({'chẵn' if i % 2 == 0 else 'lẻ'}), j={j} ({'chẵn' if j % 2 == 0 else 'lẻ'}), "
                  f"thay {before[i:j]} bằng {sub} -> {track}")
        else:
            i, j = bisect_right(track, l), bisect_left(track, r)
            print(f"query({l},{r}):  i=bisect_right={i}, j=bisect_left={j} -> {i == j and i % 2 == 1}")

trace_bisect([("add", 10, 20), ("remove", 14, 16), ("query", 10, 14), ("query", 13, 15), ("query", 16, 17)])
print()
trace_bisect([("add", 1, 3), ("add", 5, 7), ("add", 3, 5), ("query", 2, 6), ("remove", 2, 6), ("query", 1, 2), ("query", 6, 7), ("query", 2, 3)])

add(10,20):    i=0 (chẵn), j=0 (chẵn), thay [] bằng [10, 20] -> [10, 20]
remove(14,16): i=1 (lẻ), j=1 (lẻ), thay [] bằng [14, 16] -> [10, 14, 16, 20]
query(10,14):  i=bisect_right=1, j=bisect_left=1 -> True
query(13,15):  i=bisect_right=1, j=bisect_left=2 -> False
query(16,17):  i=bisect_right=3, j=bisect_left=3 -> True

add(1,3):    i=0 (chẵn), j=0 (chẵn), thay [] bằng [1, 3] -> [1, 3]
add(5,7):    i=2 (chẵn), j=2 (chẵn), thay [] bằng [5, 7] -> [1, 3, 5, 7]
add(3,5):    i=1 (lẻ), j=3 (lẻ), thay [3, 5] bằng [] -> [1, 7]
query(2,6):  i=bisect_right=1, j=bisect_left=1 -> True
remove(2,6): i=1 (lẻ), j=1 (lẻ), thay [] bằng [2, 6] -> [1, 2, 6, 7]
query(1,2):  i=bisect_right=1, j=bisect_left=1 -> True
query(6,7):  i=bisect_right=3, j=bisect_left=3 -> True
query(2,3):  i=bisect_right=2, j=bisect_left=2 -> False


### 8.3. `trace_list` — danh sách khoảng (Cách 2)

In [12]:
def trace_list(ops):
    rm = RangeModuleList()
    for op, l, r in ops:
        if op == "query":
            print(f"query({l},{r}) = {rm.queryRange(l, r)}   (khoảng: {rm.iv})")
        else:
            getattr(rm, op + "Range")(l, r)
            print(f"{op}({l},{r}) -> {rm.iv}")

trace_list([("add", 10, 20), ("remove", 14, 16), ("query", 10, 14), ("query", 13, 15),
            ("add", 14, 16), ("query", 10, 20), ("remove", 10, 20), ("query", 15, 16)])

add(10,20) -> [[10, 20]]
remove(14,16) -> [[10, 14], [16, 20]]
query(10,14) = True   (khoảng: [[10, 14], [16, 20]])
query(13,15) = False   (khoảng: [[10, 14], [16, 20]])
add(14,16) -> [[10, 20]]
query(10,20) = True   (khoảng: [[10, 20]])
remove(10,20) -> []
query(15,16) = False   (khoảng: [])


### 8.4. Đếm số nút của Segment Tree động

In [13]:
def count_nodes(node):
    if node is None:
        return 0
    return 1 + count_nodes(node.l) + count_nodes(node.r)

st = RangeModuleSegTree()
for op, l, r in [("add", 10, 20), ("remove", 14, 16), ("add", 1, 10**9), ("remove", 500, 600)]:
    getattr(st, op + "Range")(l, r)
    print(f"{op}({l},{r}): số nút = {count_nodes(st.root)}")
print("query(1, 500):", st.queryRange(1, 500), "| query(499, 501):", st.queryRange(499, 501))

add(10,20): số nút = 65
remove(14,16): số nút = 69
add(1,1000000000): số nút = 59
remove(500,600): số nút = 129
query(1, 500): True | query(499, 501): False


Chú ý `add(1, 10^9)` phủ các ô `1 .. 10^9 - 1` → mọi nút có đoạn nằm trọn trong vùng này bị thu gọn (bỏ con), nên số nút **giảm** thay vì tăng; chỉ còn lại một đường đi xuống ô cuối `10^9` (nằm ngoài vùng phủ). Mỗi lệnh tạo thêm tối đa khoảng `2 × 30` nút — đúng với đánh giá `O(log C)`.

## 9. Code nộp LeetCode

Chọn **Cách 3** (mốc phẳng + `bisect`): ngắn, nhanh, ít lỗi khi đã hiểu chẵn/lẻ.

In [14]:
from bisect import bisect_left, bisect_right

class RangeModule:

    def __init__(self):
        self.track = []

    def addRange(self, left: int, right: int) -> None:
        i = bisect_left(self.track, left)
        j = bisect_right(self.track, right)
        sub = []
        if i % 2 == 0:
            sub.append(left)
        if j % 2 == 0:
            sub.append(right)
        self.track[i:j] = sub

    def queryRange(self, left: int, right: int) -> bool:
        i = bisect_right(self.track, left)
        j = bisect_left(self.track, right)
        return i == j and i % 2 == 1

    def removeRange(self, left: int, right: int) -> None:
        i = bisect_left(self.track, left)
        j = bisect_right(self.track, right)
        sub = []
        if i % 2 == 1:
            sub.append(left)
        if j % 2 == 1:
            sub.append(right)
        self.track[i:j] = sub

assert run(RangeModule, EX_OPS, EX_ARGS) == EX_OUT
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bất biến (Cách 3).** `track` tăng **nghiêm ngặt**, độ dài chẵn, và tập được theo dõi = `⋃ [track[2t], track[2t+1])`. Hệ quả: điểm `x` được theo dõi ⇔ `bisect_right(track, x)` lẻ.

**addRange(left, right).** Tập mới = tập cũ ∪ `[left, right)`.

- Mọi mốc trong `track[i:j]` (tức mốc `m` với `left ≤ m ≤ right`) nằm trong `[left, right]`; sau khi hợp, chúng không còn là điểm "chuyển trạng thái" (bên trái và bên phải mỗi mốc đều được theo dõi, hoặc mốc trùng mút nên không cần) → xoá hết.
- Bên trái `left`: trạng thái ngay trước `left` là "được theo dõi" ⇔ `i` (số mốc `< left`) lẻ. Nếu lẻ, khoảng cũ kéo dài liền vào khoảng mới → không cần mốc `left`. Nếu chẵn → cần mốc mở `left`.
- Bên phải `right`: trạng thái ngay sau `right` là "được theo dõi" ⇔ `j` (số mốc `≤ right`) lẻ. Nếu lẻ → khoảng cũ nối tiếp → không cần mốc `right`; nếu chẵn → cần mốc đóng `right`.

Kết quả vẫn tăng nghiêm ngặt và chẵn mốc, mô tả đúng tập mới. **removeRange** chứng minh đối xứng (thay "được theo dõi" bằng "không").

**queryRange(left, right).** `i = #mốc ≤ left`, `j = #mốc < right`. `i == j` ⇔ không có mốc trong `(left, right)` ⇔ trạng thái hằng trên `[left, right)`; trạng thái đó = trạng thái tại `left` = "được theo dõi" ⇔ `i` lẻ.

**Cách 2** giữ bất biến "các khoảng rời nhau, tăng dần, giữa hai khoảng liên tiếp có khe dương" → một truy vấn được phủ kín ⇔ nằm trong một khoảng. **Cách 4** đúng theo cấu trúc Segment Tree (mục 7.3).

## 11. Phân tích Time/Space

| Cách | add/remove | query | Space |
|---|---|---|---|
| 1. Tập ô | `O(r - l)` | `O(r - l)` | `O(tổng độ dài)` |
| 2. List khoảng | `O(k)` | `O(log k)` | `O(k)` |
| 3. Mốc phẳng | `O(log k + k)` (gán lát cắt) | `O(log k)` | `O(k)` |
| 4. Segment Tree động | `O(log C)` | `O(log C)` | `O(q log C)` |

Đo với `10^4` lệnh ngẫu nhiên, toạ độ tới `10^9`:

In [15]:
import random, time

random.seed(715)
big_ops = []
for _ in range(10000):
    l = random.randint(1, 10**9 - 1)
    r = min(10**9, l + random.randint(1, 10**6))
    big_ops.append((random.choice(["addRange", "removeRange", "queryRange"]), l, r))

results = {}
for cls in [RangeModuleList, RangeModuleBisect, RangeModuleSegTree]:
    t0 = time.perf_counter()
    obj = cls()
    res = [getattr(obj, op)(l, r) for op, l, r in big_ops]
    results[cls.__name__] = res
    print(f"{cls.__name__:<20}: {time.perf_counter() - t0:.3f}s, số query true = {sum(1 for x in res if x)}")
assert results["RangeModuleList"] == results["RangeModuleBisect"] == results["RangeModuleSegTree"]

RangeModuleList     : 0.280s, số query true = 535
RangeModuleBisect   : 0.007s, số query true = 535
RangeModuleSegTree  : 0.164s, số query true = 535


## 12. Bộ test

1. Ví dụ đề.
2. Edge case: query trên module rỗng; gộp các khoảng chạm nhau; xoá ở giữa (tách đôi); xoá chạm mép (không ảnh hưởng); add trùng lặp; remove phủ nhiều khoảng; query đúng tại mút.
3. Ngẫu nhiên có seed (toạ độ 1..30): mọi cách so với brute force — so cả kết quả query và **tập ô** được phủ sau mỗi lệnh.

In [16]:
CLASSES = [RangeModuleList, RangeModuleBisect, RangeModuleSegTree, RangeModule]

def covered_cells(obj, lo=1, hi=31):
    return [x for x in range(lo, hi) if obj.queryRange(x, x + 1)]

def check_ops(ops, expected=None):
    objs = [RangeModuleBrute()] + [c() for c in CLASSES]
    outs = [[] for _ in objs]
    for op, l, r in ops:
        for k, o in enumerate(objs):
            outs[k].append(getattr(o, op)(l, r))
    for k in range(1, len(objs)):
        assert outs[k] == outs[0], (ops, CLASSES[k - 1].__name__, outs[k], outs[0])
    if expected is not None:
        got = [x for (op, _, _), x in zip(ops, outs[0]) if op == "queryRange"]
        assert got == expected, (got, expected)
    return outs[0]

# 1. Ví dụ đề
for c in CLASSES + [RangeModuleBrute]:
    assert run(c, EX_OPS, EX_ARGS) == EX_OUT

# 2. Edge case
check_ops([("queryRange", 1, 2)], [False])
check_ops([("addRange", 1, 3), ("addRange", 3, 5), ("queryRange", 1, 5)], [True])        # gộp chạm
check_ops([("addRange", 1, 10), ("removeRange", 4, 6), ("queryRange", 1, 4), ("queryRange", 6, 10),
           ("queryRange", 3, 5), ("queryRange", 5, 6)], [True, True, False, False])         # tách đôi
check_ops([("addRange", 5, 8), ("removeRange", 1, 5), ("removeRange", 8, 9), ("queryRange", 5, 8)], [True])  # xoá chạm mép
check_ops([("addRange", 2, 6), ("addRange", 2, 6), ("addRange", 3, 4), ("queryRange", 2, 6)], [True])
check_ops([("addRange", 1, 2), ("addRange", 4, 5), ("addRange", 7, 8), ("removeRange", 1, 8),
           ("queryRange", 1, 2), ("queryRange", 4, 5)], [False, False])
check_ops([("addRange", 10, 20), ("queryRange", 19, 20), ("queryRange", 20, 21), ("queryRange", 9, 10)], [True, False, False])
print("Ví dụ + edge case: OK")

# 3. Ngẫu nhiên có seed
random.seed(2031)
for _ in range(400):
    ops = []
    for _ in range(random.randint(1, 25)):
        l = random.randint(1, 29)
        r = random.randint(l + 1, 30)
        ops.append((random.choice(["addRange", "removeRange", "queryRange"]), l, r))
    check_ops(ops)
    objs = [RangeModuleBrute()] + [c() for c in CLASSES]
    for op, l, r in ops:
        for o in objs:
            getattr(o, op)(l, r)
    ref = covered_cells(objs[0])
    assert all(covered_cells(o) == ref for o in objs[1:])
print("400 chuỗi lệnh ngẫu nhiên: mọi cách khớp brute force (kết quả query và tập phủ)")

Ví dụ + edge case: OK


400 chuỗi lệnh ngẫu nhiên: mọi cách khớp brute force (kết quả query và tập phủ)


## 13. Lỗi người mới hay gặp

### Lỗi 1: Không gộp các khoảng **chạm nhau** khi thêm

In [17]:
class RangeModuleNoTouchMerge(RangeModuleList):
    def addRange(self, left, right):
        res, placed = [], False
        for a, b in self.iv:
            if b <= left:                              # SAI: chạm (b == left) cũng bị coi là rời
                res.append([a, b])
            elif right <= a:                           # SAI tương tự
                if not placed:
                    res.append([left, right]); placed = True
                res.append([a, b])
            else:
                left, right = min(left, a), max(right, b)
        if not placed:
            res.append([left, right])
        self.iv = res

for cls in [RangeModuleNoTouchMerge, RangeModuleList]:
    rm = cls()
    rm.addRange(1, 3); rm.addRange(3, 5)
    print(f"{cls.__name__:<24}: khoảng {rm.iv}, query(2,4) = {rm.queryRange(2, 4)}")

RangeModuleNoTouchMerge : khoảng [[1, 3], [3, 5]], query(2,4) = False
RangeModuleList         : khoảng [[1, 5]], query(2,4) = True


**Vì sao sai:** `[1,3)` và `[3,5)` hợp lại là `[1,5)`, nhưng nếu lưu thành hai khoảng thì truy vấn "một khoảng chứa trọn `[2,4)`" thất bại. Khi thêm, phải gộp cả khi **chạm**: giữ riêng chỉ khi `b < left` hoặc `right < a`.

### Lỗi 2: Đổi `bisect_right` thành `bisect_left` trong `queryRange`

In [18]:
class RangeModuleBadQuery(RangeModuleBisect):
    def queryRange(self, left, right):
        i = bisect_left(self.track, left)              # SAI: phải là bisect_right
        j = bisect_left(self.track, right)
        return i == j and i % 2 == 1

for cls in [RangeModuleBadQuery, RangeModuleBisect]:
    rm = cls()
    rm.addRange(10, 20)
    print(f"{cls.__name__:<20}: query(10, 14) = {rm.queryRange(10, 14)}")

RangeModuleBadQuery : query(10, 14) = False
RangeModuleBisect   : query(10, 14) = True


**Vì sao sai:** `left = 10` trùng mốc mở. `bisect_left` không đếm mốc `10` → `i = 0` (chẵn) → báo "không theo dõi", trong khi điểm 10 **thuộc** `[10, 20)`. Trạng thái tại `x` được xác định bởi số mốc **≤ x** → `bisect_right`.

### Lỗi 3: `removeRange` quên phần còn lại bên phải khi xoá ở giữa

In [19]:
class RangeModuleBadRemove(RangeModuleList):
    def removeRange(self, left, right):
        res = []
        for a, b in self.iv:
            if b <= left or right <= a:
                res.append([a, b])
            elif a < left:
                res.append([a, left])                  # SAI: dùng elif + quên [right, b)
        self.iv = res

for cls in [RangeModuleBadRemove, RangeModuleList]:
    rm = cls()
    rm.addRange(10, 20); rm.removeRange(14, 16)
    print(f"{cls.__name__:<20}: khoảng {rm.iv}, query(16, 17) = {rm.queryRange(16, 17)}")

RangeModuleBadRemove: khoảng [[10, 14]], query(16, 17) = False
RangeModuleList     : khoảng [[10, 14], [16, 20]], query(16, 17) = True


**Vì sao sai:** xoá `[14, 16)` khỏi `[10, 20)` phải để lại **hai** mảnh `[10, 14)` và `[16, 20)`. Hai điều kiện "còn phần trái" và "còn phần phải" độc lập → dùng hai `if` riêng.

### Lỗi 4: Dùng mảng đánh dấu `bool[10^9]`

Một tỷ phần tử → hết bộ nhớ. Toạ độ lớn + ít thao tác → lưu **mút** (Cách 2/3) hoặc Segment Tree **động** (Cách 4).

## 14. Mẹo & nhận dạng pattern

- **"Thêm/xoá/hỏi trên khoảng, toạ độ lớn"** → tập khoảng rời nhau đã sắp xếp (hoặc Segment Tree động).
- **Danh sách mốc phẳng + chẵn/lẻ**: điểm `x` thuộc tập ⇔ `bisect_right(track, x)` lẻ. Thêm/xoá = thay `track[i:j]` bằng 0–2 mốc.
- **Khoảng nửa mở**: khi **thêm** thì gộp cả khi chạm; khi **xoá**/**giao** thì chạm không ảnh hưởng.
- Bài cùng họ: *Merge Intervals* (56), *Insert Interval* (57), *My Calendar I/II/III* (729/731/732), *Data Stream as Disjoint Intervals* (352), *Falling Squares* (699).

## 15. Biến thể

1. **Tổng độ dài đang được theo dõi** → thêm `total` cập nhật khi thay `track[i:j]` (Cách 3), hoặc lưu tổng ở nút Segment Tree.
2. **Đếm số điểm nguyên được theo dõi trong `[l, r)`** → Segment Tree lưu tổng.
3. **Query "có giao" thay vì "phủ kín"** → `queryAny(l, r)`: có khoảng nào giao `[l, r)` không.
4. **Toggle (đảo trạng thái) một đoạn** → Segment Tree với lazy "đảo bit".

Ví dụ biến thể 1 và 3 trên Cách 3:

In [20]:
class RangeModulePlus(RangeModuleBisect):
    def total_length(self):
        t = self.track
        return sum(t[k + 1] - t[k] for k in range(0, len(t), 2))

    def query_any(self, left, right):
        i = bisect_right(self.track, left)
        j = bisect_left(self.track, right)
        return i % 2 == 1 or i != j          # left đang được theo dõi, hoặc có mốc nằm trong (left, right)

rm = RangeModulePlus()
rm.addRange(10, 20); rm.removeRange(14, 16); rm.addRange(30, 35)
print("track:", rm.track, "| tổng độ dài:", rm.total_length())
print("query_any(14,16):", rm.query_any(14, 16), "| query_any(15,17):", rm.query_any(15, 17), "| query_any(20,30):", rm.query_any(20, 30))

random.seed(352)
for _ in range(300):
    rm, br = RangeModulePlus(), RangeModuleBrute()
    for _ in range(15):
        l = random.randint(1, 19); r = random.randint(l + 1, 20)
        op = random.choice(["addRange", "removeRange"])
        getattr(rm, op)(l, r); getattr(br, op)(l, r)
    assert rm.total_length() == len(br.cells)
    l = random.randint(1, 19); r = random.randint(l + 1, 20)
    assert rm.query_any(l, r) == any(x in br.cells for x in range(l, r))
print("biến thể khớp brute force")

track: [10, 14, 16, 20, 30, 35] | tổng độ dài: 13
query_any(14,16): False | query_any(15,17): True | query_any(20,30): False
biến thể khớp brute force


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tập ô | Lưu từng ô đơn vị | `O(r - l)` mỗi lệnh | `O(tổng độ dài)` | Trọng tài khi test, toạ độ nhỏ |
| 2. List khoảng | Gộp/cắt bằng duyệt tuyến tính; query nhị phân | `O(k)` / `O(log k)` | `O(k)` | Dễ hiểu, dễ mở rộng |
| 3. Mốc phẳng | `bisect` + chẵn/lẻ, thay lát cắt | `O(log k + k)` / `O(log k)` | `O(k)` | Lời giải ngắn, nhanh (khuyên dùng) |
| 4. Segment Tree động | Nút đồng nhất + tách khi cần | `O(log C)` | `O(q log C)` | Cần truy vấn phức tạp hơn (tổng, đếm) |

## 17. Bài tập

**Bài 1 (đoán output).** Sau `add(1,5)`, `remove(2,3)`, `add(3,4)`: `query(1,2)`, `query(1,3)`, `query(3,5)`?

**Bài 2 (tính tay).** Lập bảng `track`, `i`, `j` (như 8.1) cho chuỗi `add(5,10)`, `add(1,3)`, `add(3,5)`, `remove(4,6)`.

**Bài 3 (điền TODO).** Hoàn thiện `add_track(track, left, right)` trả về `track` mới (Cách 3).

**Bài 4 (tự viết lại).** Viết lại lớp `MyRangeModule` theo Cách 2 (list khoảng).

**Bài 5 (mini challenge).** Thêm `count_gaps(left, right)` vào Cách 3: trả về số đoạn **không** được theo dõi (rời nhau, độ dài dương) nằm trong `[left, right)`.

In [21]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def add_track(track, left, right):
    # TODO: i = bisect_left(track, left); j = bisect_right(track, right);
    #       sub = [left nếu i chẵn] + [right nếu j chẵn]; trả track[:i] + sub + track[j:]
    pass

# Bài 4 — TODO
class MyRangeModule:
    def __init__(self):
        # TODO: self.iv = []
        pass
    def addRange(self, left, right):
        # TODO: gộp các khoảng chồng/chạm
        pass
    def queryRange(self, left, right):
        # TODO: tìm khoảng có a <= left lớn nhất
        pass
    def removeRange(self, left, right):
        # TODO: giữ phần trái [a, left) và phần phải [right, b)
        pass

# Bài 5 — TODO
def count_gaps(track, left, right):
    # TODO: lấy các mốc trong (left, right), thêm left/right làm biên, đếm các đoạn không theo dõi
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: theo dõi danh sách khoảng sau mỗi lệnh.
- Mức 2: sau ba lệnh: `[1,2)`, `[3,5)`.
- Mức 3: `True`, `False`, `True`.

**Bài 2.**
- Mức 1: `add(5,10)`: `[5,10]`.
- Mức 2: `add(1,3)`: `[1,3,5,10]`; `add(3,5)`: `i = bisect_left(3) = 1` (lẻ), `j = bisect_right(5) = 3` (lẻ) → thay `[3,5]` bằng `[]` → `[1,10]`.
- Mức 3: `remove(4,6)`: `i = 1` (lẻ), `j = 1` (lẻ) → chèn `[4,6]` → `[1,4,6,10]`.

**Bài 3.**
- Mức 1: tính `i`, `j` bằng hai hàm bisect khác nhau.
- Mức 2: chẵn → thêm mốc.
- Mức 3: ghép `track[:i] + sub + track[j:]`.

**Bài 4.**
- Mức 1: add: giữ khi `b < left` hoặc `right < a`.
- Mức 2: remove: giữ khi `b <= left` hoặc `right <= a`.
- Mức 3: query: `bisect_right(iv, [left, inf]) - 1`.

**Bài 5.**
- Mức 1: các đoạn giữa những điểm `left`, các mốc trong `(left, right)`, `right`.
- Mức 2: mỗi đoạn được theo dõi hay không xác định bởi `bisect_right(track, điểm đầu đoạn)` lẻ/chẵn.
- Mức 3: đếm đoạn không theo dõi (các đoạn liên tiếp đã tự xen kẽ nên không cần gộp).

## 19. Đáp án tham khảo

In [22]:
# Bài 1
rm = RangeModule()
rm.addRange(1, 5); rm.removeRange(2, 3); rm.addRange(3, 4)
assert (rm.queryRange(1, 2), rm.queryRange(1, 3), rm.queryRange(3, 5)) == (True, False, True)

# Bài 2
rm = RangeModule()
rm.addRange(5, 10); assert rm.track == [5, 10]
rm.addRange(1, 3);  assert rm.track == [1, 3, 5, 10]
rm.addRange(3, 5);  assert rm.track == [1, 10]
rm.removeRange(4, 6); assert rm.track == [1, 4, 6, 10]

# Bài 3
def add_track_ans(track, left, right):
    i, j = bisect_left(track, left), bisect_right(track, right)
    sub = [left] * (i % 2 == 0) + [right] * (j % 2 == 0)
    return track[:i] + sub + track[j:]

assert add_track_ans([1, 3, 5, 10], 3, 5) == [1, 10]
assert add_track_ans([], 2, 4) == [2, 4]
print("Bài 1–3 OK")

Bài 1–3 OK


In [23]:
# Bài 4
class MyRangeModuleAns:
    def __init__(self):
        self.iv = []
    def addRange(self, left, right):
        keep_l = [iv for iv in self.iv if iv[1] < left]
        keep_r = [iv for iv in self.iv if iv[0] > right]
        mid = [iv for iv in self.iv if not (iv[1] < left or iv[0] > right)]
        if mid:
            left, right = min(left, mid[0][0]), max(right, mid[-1][1])
        self.iv = keep_l + [[left, right]] + keep_r
    def queryRange(self, left, right):
        k = bisect_right(self.iv, [left, float('inf')]) - 1
        return k >= 0 and self.iv[k][0] <= left and right <= self.iv[k][1]
    def removeRange(self, left, right):
        res = []
        for a, b in self.iv:
            if b <= left or right <= a:
                res.append([a, b])
            else:
                if a < left: res.append([a, left])
                if right < b: res.append([right, b])
        self.iv = res

# Bài 5
def count_gaps_ans(track, left, right):
    pts = [left] + [m for m in track if left < m < right] + [right]
    return sum(1 for a in pts[:-1] if bisect_right(track, a) % 2 == 0)

def count_gaps_brute(cells, left, right):
    gaps, inside = 0, False
    for x in range(left, right):
        if x not in cells and not inside:
            gaps += 1
        inside = x not in cells
    return gaps

random.seed(77)
for _ in range(400):
    a, b, c = MyRangeModuleAns(), RangeModuleBrute(), RangeModule()
    for _ in range(random.randint(1, 15)):
        l = random.randint(1, 19); r = random.randint(l + 1, 20)
        op = random.choice(["addRange", "removeRange", "queryRange"])
        assert getattr(a, op)(l, r) == getattr(b, op)(l, r) == getattr(c, op)(l, r)
    l = random.randint(1, 19); r = random.randint(l + 1, 20)
    assert count_gaps_ans(c.track, l, r) == count_gaps_brute(b.cells, l, r)
print("Bài 4–5 OK (đối chiếu brute force)")

Bài 4–5 OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template mốc phẳng (nhớ bảng chẵn/lẻ):**

```python
add:    i = bisect_left(t, L);  j = bisect_right(t, R); t[i:j] = [L]*(i%2==0) + [R]*(j%2==0)
remove: i = bisect_left(t, L);  j = bisect_right(t, R); t[i:j] = [L]*(i%2==1) + [R]*(j%2==1)
query:  i = bisect_right(t, L); j = bisect_left(t, R);  return i == j and i % 2 == 1
```

**Template list khoảng:** add giữ khi `b < L or R < a` (còn lại gộp); remove giữ khi `b <= L or R <= a` (còn lại cắt hai phía).

**Checklist trước khi submit:**

- [ ] Khoảng nửa mở `[left, right)`; thêm thì **gộp cả khi chạm**.
- [ ] Remove tách đúng **hai** mảnh khi xoá ở giữa.
- [ ] Query: `bisect_right` cho `left`, `bisect_left` cho `right`.
- [ ] Add/remove: `bisect_left` cho `left`, `bisect_right` cho `right`; chẵn/lẻ đúng bảng.
- [ ] Không dùng mảng kích thước `10^9`.
- [ ] Test ví dụ đề + chạm mép + query tại mút.
- [ ] Tên lớp/method đúng: `RangeModule`, `addRange`, `queryRange`, `removeRange`.